## Cell 0 — Runtime probe and installation

In [ ]:
# =============================================================
# CELL 0 — Runtime probe + dependencies
# =============================================================
import os, sys, subprocess, platform, textwrap

def sh(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=False)

# ---- 0.1 Probe the machine we were actually given -------------
import multiprocessing
print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())
print("CPU cores   :", multiprocessing.cpu_count(), "  <-- drives ORC runtime (see PRD 5.1)")
try:
    import psutil
    print("System RAM  : %.1f GB" % (psutil.virtual_memory().total / 1e9))
except ImportError:
    sh("cat /proc/meminfo | head -1")

try:
    import torch
    print("Torch       :", torch.__version__, "| CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print("GPU         :", p.name, "| VRAM: %.1f GB" % (p.total_memory / 1e9))
except ImportError:
    print("Torch not present yet.")

# ---- 0.2 Install ---------------------------------------------
sh("pip install -q GraphRicciCurvature POT networkx>=3.4 pyarrow tqdm")
sh("pip install -q torch-geometric")
sh("pip install -q transformers sentence-transformers adapters")

# ---- 0.3 Drive for checkpoints -------------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_ROOT = '/content/drive/MyDrive/orca_gcn_caif'
except Exception:
    CACHE_ROOT = './orca_gcn_caif'          # local / CI fallback
os.makedirs(CACHE_ROOT, exist_ok=True)
print("Cache root  :", CACHE_ROOT)


## Cell 1 — Master configuration

In [ ]:
# =============================================================
# CELL 1 — CFG: the single source of every tunable value
# =============================================================
from dataclasses import dataclass, asdict, field
from typing import Optional, Tuple
import json, hashlib, os

@dataclass
class Config:
    # ---------- Phase 1: acquisition ----------
    OPENALEX_API_KEY: str = ""             # REQUIRED since 2026-02-13
    OPENALEX_BASE: str = "https://api.openalex.org"
    JOURNAL_IDS: Tuple[str, ...] = ("S137773608",)   # OpenAlex source IDs
    YEAR_START: int = 2022
    YEAR_END: int = 2023
    MAX_SEED_PAPERS: int = 20000
    CITES_BATCH: int = 50                  # OR-filter IDs per request
    PER_PAGE: int = 200
    REQUEST_TIMEOUT: int = 30
    MAX_RETRIES: int = 5
    SELECT_FIELDS: str = ("id,doi,title,publication_year,publication_date,"
                          "primary_location,authorships,referenced_works,"
                          "topics,abstract_inverted_index,cited_by_count")

    # ---------- Phase 2a: ORC ----------
    ORC_ALPHA: float = 0.5                 # random-walk laziness
    ORC_METHOD: str = "OTDSinkhornMix"     # OTD | Sinkhorn | OTDSinkhornMix
    ORC_PROC_CAP: int = 12
    ORC_ISOLATED_VALUE: float = 0.0

    # ---------- Phase 2b: ORCA-GCN ----------
    TAU_CURV: float = 1.0
    GATE_MODE: str = "softplus"            # softplus | exp | sigmoid
    GCN_LAYERS: int = 2
    GCN_HIDDEN: int = 128
    GCN_DROPOUT: float = 0.5
    GCN_LR: float = 0.01
    GCN_EPOCHS: int = 200
    GCN_USE_TEXT_FEATURES: bool = True
    GCN_TEXT_PCA_DIM: int = 64

    # ---------- Phase 2c: temporal ----------
    LAMBDA_DECAY: Optional[float] = None   # None => MLE-fit per field (PRD 4.3)
    TEMPORAL_IMPOSSIBLE_VALUE: float = 0.0

    # ---------- Phase 2d: semantic ----------
    SEMANTIC_BACKEND: str = "specter2"     # specter2 | st-specter | scibert
    EMB_BATCH_SIZE: int = 32
    EMB_MAX_TOKENS: int = 512
    EMB_FP16: bool = True
    SEMANTIC_MISSING_VALUE: float = 0.5

    # ---------- Phase 2e: fusion ----------
    STANDARDIZE: str = "robust"            # robust | zscore | none
    FUSION_ORDER: Tuple[str, ...] = ("kappa", "h", "tau", "sigma")

    # ---------- Phase 3: trust head ----------
    LOGIT_CLAMP: float = 30.0              # PRD 2.3.3 — Test 11 depends on this
    HEAD_LR: float = 0.01
    HEAD_L2: float = 1e-4
    HEAD_EPOCHS: int = 300
    HEAD_POS_WEIGHT: Optional[float] = None   # None => auto from class balance
    WEAK_CARTEL_PCT: float = 0.99
    WEAK_AUTHOR_OVERLAP: float = 0.50
    WEAK_BULK_REF_MIN: int = 5
    WEAK_POS_MIN_CITED: int = 10

    # ---------- Phase 4: metric + validation ----------
    FIELD_PEER_K: int = 20
    NULL_MODE: str = "strict"              # fast | strict
    NULL_B_FAST: int = 500
    NULL_B_STRICT: int = 50
    NULL_SWAP_FACTOR: int = 10
    Z_THRESHOLD: float = 1.96
    RECAL_VAR_TRIGGER: float = 0.30
    RECAL_MAX_PASSES: int = 2

    # ---------- infrastructure ----------
    SEED: int = 42
    USE_CACHE: bool = True
    CACHE_ROOT: str = CACHE_ROOT
    LOG_LEVEL: str = "INFO"

CFG = Config()

def _resolve_api_key():
    """Colab Secrets first (never paste a key into a cell or chat), then
    the OPENALEX_API_KEY env var for local/CI runs. See README.md."""
    try:
        from google.colab import userdata
        key = userdata.get("OPENALEX_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return os.environ.get("OPENALEX_API_KEY", "")

CFG.OPENALEX_API_KEY = _resolve_api_key()
assert CFG.OPENALEX_API_KEY, (
    "No OpenAlex API key found. In Colab: Secrets panel (key icon, left "
    "sidebar) -> add OPENALEX_API_KEY -> enable notebook access. "
    "Locally: set the OPENALEX_API_KEY env var. Get a free key at "
    "openalex.org/settings/api (PRD 3.5)."
)

def cfg_hash(c) -> str:
    return hashlib.sha256(json.dumps(asdict(c), sort_keys=True,
                                     default=str).encode()).hexdigest()[:12]

CFG_HASH = cfg_hash(CFG)
print("Config hash:", CFG_HASH)
with open(os.path.join(CFG.CACHE_ROOT, f"cfg_{CFG_HASH}.json"), "w") as f:
    json.dump(asdict(CFG), f, indent=2, default=str)


## Cell 2 — Utilities: cache, retry, determinism

In [ ]:
# =============================================================
# CELL 2 — Infrastructure utilities
# =============================================================
import os, gzip, pickle, random, logging, time, functools
import numpy as np, pandas as pd

logging.basicConfig(level=getattr(logging, CFG.LOG_LEVEL),
                    format="%(asctime)s | %(levelname)-7s | %(message)s")
log = logging.getLogger("caif")

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
    except ImportError:
        pass
set_seed(CFG.SEED)

# ---- checkpoint helpers: every expensive cell uses these ------
# pickle is used only for this notebook's own Drive cache (CFG.CACHE_ROOT):
# files this same run wrote and reads back, never an externally-sourced
# artifact. Not a deserialization-of-untrusted-input risk in this context.
def _p(name, ext): return os.path.join(CFG.CACHE_ROOT, f"{name}.{ext}")

def cache_exists(name, ext="pkl"): return CFG.USE_CACHE and os.path.exists(_p(name, ext))

def cache_save(name, obj, ext="pkl"):
    with open(_p(name, ext), "wb") as f: pickle.dump(obj, f, protocol=4)
    log.info("cached -> %s", _p(name, ext))

def cache_load(name, ext="pkl"):
    with open(_p(name, ext), "rb") as f: obj = pickle.load(f)
    log.info("loaded <- %s", _p(name, ext)); return obj

def stage(name, ext="pkl"):
    """Decorator: skip an expensive stage entirely if its artifact exists."""
    def deco(fn):
        @functools.wraps(fn)
        def wrap(*a, **kw):
            if cache_exists(name, ext): return cache_load(name, ext)
            t0 = time.time(); out = fn(*a, **kw)
            log.info("stage %-12s took %.1fs", name, time.time() - t0)
            cache_save(name, out, ext); return out
        return wrap
    return deco

def safe_finite(x, nan=0.0, posinf=1.0, neginf=-1.0):
    """Test 03: no NaN / Inf may ever enter the feature vector."""
    return np.nan_to_num(np.asarray(x, dtype=np.float64),
                         nan=nan, posinf=posinf, neginf=neginf)
